In [1]:
with open('input.txt','r', encoding='utf-8') as f:
    text = f.read()

In [2]:
len(text)

1115394

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
''.join(chars)

65

In [13]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode("Hello world!"))
print(decode([39, 50, 50, 39, 46, 1, 39, 49, 40, 39, 56]))

[20, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42, 2]
allah akbar


In [17]:
import torch
data = torch.tensor(encode(text),dtype=torch.long)

In [33]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"contexte:{context} pour targe: {target}")

contexte:tensor([18]) pour targe: 47
contexte:tensor([18, 47]) pour targe: 56
contexte:tensor([18, 47, 56]) pour targe: 57
contexte:tensor([18, 47, 56, 57]) pour targe: 58
contexte:tensor([18, 47, 56, 57, 58]) pour targe: 1
contexte:tensor([18, 47, 56, 57, 58,  1]) pour targe: 15
contexte:tensor([18, 47, 56, 57, 58,  1, 15]) pour targe: 47
contexte:tensor([18, 47, 56, 57, 58,  1, 15, 47]) pour targe: 58


tensor([], dtype=torch.int64)

In [69]:
torch.manual_seed(1337)
batch_size = 4 #independant sequences in parralel
block_size = 8 # what is the maximum context length for predictions ?
#train_data1 = data[:15]
def get_batch(split):
    data = train_data if split =='train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs: ',xb)
print('targets: ', yb)
print(len(data))
b = 0
for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(f'context: {context.tolist()} with target: {target}')


inputs:  tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:  tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
1115394
context: [24] with target: 43
context: [24, 43] with target: 58
context: [24, 43, 58] with target: 5
context: [24, 43, 58, 5] with target: 57
context: [24, 43, 58, 5, 57] with target: 1
context: [24, 43, 58, 5, 57, 1] with target: 46
context: [24, 43, 58, 5, 57, 1, 46] with target: 43
context: [24, 43, 58, 5, 57, 1, 46, 43] with target: 39


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)
class BigramLanguageMode(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token reads off the logits for the next token f lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B,T,C)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
m = BigramLanguageMode(vocab_size)
logits, loss = m(xb, yb)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

tensor(4.8786, grad_fn=<NllLossBackward0>) torch.Size([32, 65])

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [76]:
optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)

In [77]:
batch_size = 32
for steps in range(100):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    print(loss.item())

2.4114136695861816
2.4820094108581543
2.5882177352905273
2.493046522140503
2.4700257778167725
2.488373041152954
2.5371649265289307
2.4152002334594727
2.435832977294922
2.443141460418701
2.3946211338043213
2.4614155292510986
2.385409116744995
2.4698052406311035
2.629945993423462
2.4081544876098633
2.6715445518493652
2.48933482170105
2.3953559398651123
2.4484059810638428
2.555893898010254
2.477600336074829
2.383572816848755
2.389721632003784
2.506155490875244
2.560096263885498
2.5177223682403564
2.350615978240967
2.4649124145507812
2.5018255710601807
2.4960107803344727
2.5101046562194824
2.516814708709717
2.4551169872283936
2.5093274116516113
2.4393439292907715
2.421462059020996
2.536351442337036
2.3698203563690186
2.4139678478240967
2.3815696239471436
2.4839346408843994
2.603696584701538
2.557237148284912
2.4688215255737305
2.633723020553589
2.510784149169922
2.457509756088257
2.591402530670166
2.3322315216064453
2.4451160430908203
2.346325635910034
2.5013108253479004
2.598198890686035


In [80]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))




WITHat iPOUNo tthe titin


Anonce bre mpree gacho wn t it engeyodolanus
ASashoucert,
TMartheno tous t sshe nanth lsthenow d hef
ce s fr fot w y ban yourecend bo s
He ase anghavais hirthe h dzert d ORour int t ant:
Tome.
Yo pr oliso y; athoyor areser d s stordavee t
TA pag moshenothil!
IORDED dr mee tte indoup, wityont CARI RSO mmelan, e prslterar
dik,
Weashoomancte y hewhrd.
Has.

IVIOFourd
DG nd gas pryoterss me
T:

Loyo ind w yon
Gomeme ansce:
AMADo pus
LLAMPRIce my cr IIOnfadods t thiner TIO


In [100]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn((B,T,C))
x

tensor([[[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]],

        [[ 1.3488, -0.1396],
         [ 0.2858,  0.9651],
         [-2.0371,  0.4931],
         [ 1.4870,  0.5910],
         [ 0.1260, -1.5627],
         [-1.1601, -0.3348],
         [ 0.4478, -0.8016],
         [ 1.5236,  2.5086]],

        [[-0.6631, -0.2513],
         [ 1.0101,  0.1215],
         [ 0.1584,  1.1340],
         [-1.1539, -0.2984],
         [-0.5075, -0.9239],
         [ 0.5467, -1.4948],
         [-1.2057,  0.5718],
         [-0.5974, -0.6937]],

        [[ 1.6455, -0.8030],
         [ 1.3514, -0.2759],
         [-1.5108,  2.1048],
         [ 2.7630, -1.7465],
         [ 1.4516, -1.5103],
         [ 0.8212, -0.2115],
         [ 0.7789,  1.5333],
         [ 1.6097, -0.4032]]])

In [ ]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev,0)

wei = torch.tril(torch.ones(T, T))        # triang matrix
wei = wei / wei.sum(dim=1, keepdim=True)
xbow2 = wei @ x



In [ ]:
# 3 version
tril = torch.tril(torch.ones(T, T))        # triang matrix 
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow3, xbow2)

True